# Text Embedding with IndoBERTweet
**Purpose:** Convert text to dense vectors for similarity analysis & clustering

**Model:** indolem/indobertweet-base-uncased (Indonesian Twitter BERT)

**Note:** Template siap pakai ketika data baru (dengan buzzer) sudah tersedia.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Model & Data

In [ ]:
# Load IndoBERTweet
model_name = "indolem/indobertweet-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print(f"Model loaded: {model_name}")

In [ ]:
# Load data
df = pd.read_csv("../data/features/posts_with_features.csv")
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded: {len(df)} posts")
df.head()

## 3. Text Embedding

In [ ]:
def embed_text(texts, model, tokenizer, device, batch_size=16):
    """Convert texts to embeddings using IndoBERTweet"""
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            max_length=128, 
            return_tensors="pt"
        ).to(device)
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**encoded)
            # Use [CLS] token embedding (first token)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        
        embeddings.append(batch_embeddings)
    
    return np.vstack(embeddings)

# Embed body_clean (text without mentions)
texts = df["body_clean"].fillna("").tolist()
embeddings = embed_text(texts, model, tokenizer, device, batch_size=16)

print(f"Embeddings shape: {embeddings.shape}")
# ponytail: batch_size=16 for memory. Increase to 32/64 if GPU available

## 4. Similarity Analysis

In [ ]:
# Compute pairwise cosine similarity
similarity_matrix = cosine_similarity(embeddings)
print(f"Similarity matrix: {similarity_matrix.shape}")

# Find highly similar pairs (threshold = 0.95)
similar_pairs = []
for i in range(len(similarity_matrix)):
    for j in range(i+1, len(similarity_matrix)):
        if similarity_matrix[i, j] > 0.95:
            similar_pairs.append({
                "post1": df.iloc[i]["post_id"],
                "post2": df.iloc[j]["post_id"],
                "acc1": df.iloc[i]["account_id"],
                "acc2": df.iloc[j]["account_id"],
                "similarity": similarity_matrix[i, j],
                "text1": df.iloc[i]["body_clean"][:50],
                "text2": df.iloc[j]["body_clean"][:50]
            })

similar_df = pd.DataFrame(similar_pairs)
print(f"\nHighly similar pairs (>0.95): {len(similar_df)}")
if len(similar_df) > 0:
    print(similar_df.head(10))

## 5. Account Similarity

In [ ]:
# Average embedding per account
df["embedding"] = list(embeddings)
account_embeddings = df.groupby("account_id")["embedding"].apply(
    lambda x: np.mean(np.vstack(x.values), axis=0)
).reset_index()

acc_emb_matrix = np.vstack(account_embeddings["embedding"].values)
acc_similarity = cosine_similarity(acc_emb_matrix)

print(f"Account embeddings: {acc_emb_matrix.shape}")
print(f"Account similarity matrix: {acc_similarity.shape}")

# Find coordinated accounts (high similarity + different accounts)
coordinated = []
for i in range(len(acc_similarity)):
    for j in range(i+1, len(acc_similarity)):
        if acc_similarity[i, j] > 0.90:
            coordinated.append({
                "acc1": account_embeddings.iloc[i]["account_id"],
                "acc2": account_embeddings.iloc[j]["account_id"],
                "similarity": acc_similarity[i, j]
            })

coord_df = pd.DataFrame(coordinated)
print(f"\nCoordinated accounts (similarity >0.90): {len(coord_df)}")
if len(coord_df) > 0:
    print(coord_df.head(10))

## 6. Clustering (UMAP + HDBSCAN)

In [ ]:
# Install if needed: pip install umap-learn hdbscan
try:
    from umap import UMAP
    import hdbscan
    
    # Reduce dimensions
    reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    embeddings_2d = reducer.fit_transform(embeddings)
    
    # Cluster
    clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
    clusters = clusterer.fit_predict(embeddings)
    
    df["cluster"] = clusters
    df["x"] = embeddings_2d[:, 0]
    df["y"] = embeddings_2d[:, 1]
    
    print(f"Clusters found: {len(set(clusters)) - (1 if -1 in clusters else 0)}")
    print(f"Noise points: {(clusters == -1).sum()}")
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.scatter(df["x"], df["y"], c=df["cluster"], 
                        cmap="tab10", s=30, alpha=0.6)
    plt.colorbar(scatter, ax=ax, label="Cluster")
    ax.set_title("Text Embedding Clusters (UMAP + HDBSCAN)", fontsize=14, fontweight="bold")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig("../output/embedding_clusters.png", dpi=150, bbox_inches="tight")
    print("Saved: output/embedding_clusters.png")
    plt.show()
    
except ImportError:
    print("Install umap-learn and hdbscan: pip install umap-learn hdbscan")
    df["cluster"] = -1

## 7. Export

In [ ]:
# Save embeddings
np.save("../data/features/text_embeddings.npy", embeddings)
df[["post_id", "account_id", "cluster"]].to_csv("../data/features/post_clusters.csv", index=False)

print("Saved:")
print("  - data/features/text_embeddings.npy")
print("  - data/features/post_clusters.csv")
print("\nNext: Combine with network features for buzzer classification")